# SmolBench on Amazon Bedrock: a three-model ladder trio

This notebook runs both SmolBench studies on Amazon Bedrock, with reasoning switched on for
every model:

- **Periodic induction**: arms `intens`, `noise_intens` and `extens`, seeds 0–29, 9 count
  questions per replicate.
- **Horn-rule deduction**: the four paper arms `lem`, `pad`, `disc` and `both` plus the four
  review arms `fwd`, `tjunk`, `rep` and `marked-lemma`, seeds 100–199, 3 replicates,
  temperature 0.7, each model at its ICLR chain length `m`.

The three models are all in the ICLR ladder, come from three different labs, and are
served by Bedrock:

| roster key | Bedrock id | lab | parameters | Horn `m` |
|---|---|---|---|---|
| `gemma-4-31b` | `google.gemma-4-31b` (served under `/openai/v1`) | Google | 30.7B dense | 64 |
| `nemotron-3-super-120b-a12b` | `nvidia.nemotron-super-3-120b` | NVIDIA | 120B MoE, 12B active | 20 |
| `deepseek-v3.1` | resolved from `GET /models` | DeepSeek | 685B MoE, 37B active | 64 |

The steps between them are 3.9x and 5.7x, or 22x end to end, in total parameters. Bedrock
hosts no other three ladder models from different large labs. Qwen3.5 is only on
SageMaker, and Gemma-4 E2B/12B and Nemotron-3 Nano-4B are not hosted.

**Code reused.** Requests go through `smolbench.evals.providers.aws`, the OpenAI-compatible
bedrock-mantle client. The induction quizzes come from
`notebooks/induction/run_study.make_quizzes`, and the runs use `ReplicateHarness`. The
Horn rungs are rendered by `smolbench.deduction.horn.cli.write_seed` (the renderer behind
`horn.repro.render_rung`, here with all eight arms) and the paper arms are checked against
the ICLR digests; each cell is run and scored by `scripts/deduction/horn/sweep.run_cell`.
`bedrock_trio.py` adds only the roster, a per-model endpoint switch, the results store, and
the reasoning checks.

**Differences from the ICLR runs:**

- **Output caps.** Bedrock caps output, reasoning included, at 8K tokens for DeepSeek V3.1
  and 32K for Nemotron Super; Gemma 4 has no documented cap, so it is sent 32K. The ICLR
  budgets were 131K (Horn) and at least 48K (induction). A truncated answer scores
  `length` (Horn) or `null`, counted as wrong (induction). Both rates are reported below.
  The published DeepSeek and Nemotron Horn rows came from Bedrock and were capped the
  same way.
- **Induction temperature.** It is not sent, as at ICLR, but the server default here is
  Bedrock's, not the vLLM `generation_config.json` default the ICLR runs used.
- **Seeding.** Every request here is seeded: the replicate seed for induction and the
  replicate index for Horn. The published DeepSeek and Nemotron Horn rows came from
  unseeded Bedrock Converse calls; the Gemma rows and all published induction rows came
  from self-hosted vLLM.

**Running it.**
1. Refresh your AWS credentials.
2. Set `SMOKE` in the next cell. `True` runs a few seeds per model; `False` runs the full
   protocol.
3. Run all cells.

**Where results go.** The private bucket `RESULTS_S3` is the store of record; objects there
are append-only. `rebuttal/results/` is a local mirror of it that the report cells read; it
is not committed. A rerun resumes from the bucket, on this machine or any other: finished
replicates and rows are skipped. A smoke run uses the `smoke-2/` sub-prefix and
`rebuttal/results/smoke-2/`, so it never mixes with protocol runs. Rendered rungs go to `rebuttal/rungs/` (not committed,
since they are deterministic and digest-checked).

**The `extens` arm.** On this branch the `extens` context is the intensional rules followed
by the listing. The published `extens` and `noise_intens` numbers come from the earlier
`extens` arm, which carried the listing only, so only `intens` is like-for-like with the
published values.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "rebuttal" / "bedrock_trio.py").exists())
sys.path.insert(0, str(REPO_ROOT / "rebuttal"))

import bedrock_trio as bt
from smolbench.deduction.horn import repro as horn_repro
from smolbench.evals.results_store import format_run_ts, utcnow
from smolbench.induction import repro as induction_repro

SMOKE = True
REGION = "us-east-1"  # bedrock-mantle is verified here
# The store of record; None keeps results local only. A smoke run gets its own prefix and
# mirror, because the bucket is append-only and the protocol prefix must stay clean.
RESULTS_S3 = bt.RESULTS_S3 + (f"/{bt.SMOKE_PREFIX}" if SMOKE else "")
RESULTS = bt.RESULTS_DIR / bt.SMOKE_PREFIX if SMOKE else bt.RESULTS_DIR  # local mirror
HORN_ARMS = bt.HORN_ARMS  # the four paper arms, then the four review arms
MODELS = list(bt.TRIO)  # roster keys, smallest first
CONCURRENCY = 8  # parallel requests per model

if SMOKE:
    INDUCTION_SEEDS, HORN_SEEDS, HORN_REPS = [0, 1], [100, 101], 1
else:
    INDUCTION_SEEDS, HORN_SEEDS, HORN_REPS = list(range(30)), list(range(100, 200)), 3

print(f"SMOKE={SMOKE}: induction seeds {INDUCTION_SEEDS[0]}-{INDUCTION_SEEDS[-1]}, "
      f"Horn seeds {HORN_SEEDS[0]}-{HORN_SEEDS[-1]} x {HORN_REPS} reps, "
      f"results in {RESULTS_S3 or RESULTS}")

## Authenticate and resolve the model ids

This mints a short-term Bedrock bearer token from your AWS profile. An
`AWS_BEARER_TOKEN_BEDROCK` already set in the environment is used as is. When `RESULTS_S3`
is set, it also checks that the results bucket can be listed, so a credential problem fails
here and not after inference has been billed. The listing combines both mantle paths,
because Gemma 4 is only served under `/openai/v1`.

In [ ]:
bt.mint_token(REGION)
if RESULTS_S3:
    bt.check_bucket(RESULTS_S3)
listing = set()
for path in ("/v1", "/openai/v1"):
    try:
        listing |= set(bt.list_mantle_models(REGION, path))
    except Exception as err:  # one path failing is reported, not fatal
        print(f"GET {path}/models failed: {err}")
TRIO = bt.resolve_ids(sorted(listing), {k: bt.TRIO[k] for k in MODELS})
for m in TRIO.values():
    print(f"{m.key:28s} {m.bedrock_id:34s} {bt.base_url(m, REGION)}")

## Force reasoning

For each model, this sends a one-line question with each of the model's reasoning fields
in turn and keeps the first one that returns reasoning. If none does, the cell fails, and
the model is not run without reasoning. The probes are saved to `results/probes.json` and,
when `RESULTS_S3` is set, uploaded to the bucket as `probes/probes--<stamp>.json`.

In [ ]:
REASONING, PROBES = {}, {}
for key, model in TRIO.items():
    REASONING[key], PROBES[key] = bt.pick_reasoning_args(model, REGION)
    print(f"{key:28s} {REASONING[key]}  <- {PROBES[key][-1]}")
RESULTS.mkdir(parents=True, exist_ok=True)
probes_path = RESULTS / "probes.json"
probes_path.write_text(json.dumps(
    {"region": REGION, "ids": {k: m.bedrock_id for k, m in TRIO.items()},
     "reasoning_args": REASONING, "probes": PROBES}, indent=2) + "\n")
if RESULTS_S3:
    probe_key = bt.push_file(probes_path, RESULTS_S3, f"probes/probes--{format_run_ts(utcnow())}.json")
    print(f"probe record uploaded to {probe_key}")

## Check the induction prompts against the released ones

For seed 0 of each model, the regenerated prompts are compared with the prompts in the
released ICLR results, which are read anonymously from the public bucket. The tokenizers
come from Hugging Face at the pinned revisions, so the gated `google/gemma-4-31B-it` needs
`HF_TOKEN` unless it is already cached. What is checked differs by arm:

- **`intens`**: each prompt must equal the released prompt byte for byte.
- **`extens`**: on this branch the context is the intensional rules followed by the
  listing, while the released prompts carry the listing only. Each prompt, with the rules
  block removed from directly before the listing, must equal the released prompt byte for
  byte. The rules block is taken from the question's own `intens` prompt, so a rules block
  that differs from it fails.
- **`noise_intens`**: it is padded to the length of `extens`, which is longer here than in
  the release, so it cannot equal the released prompt. Its token count under the model's
  own tokenizer must equal that of the matching `extens` prompt.

The published `extens` and `noise_intens` numbers come from the earlier listing-only
`extens` arm, so only `intens` is like-for-like with the published values.

In [ ]:
run_study = bt.load_run_study()
for key in TRIO:
    bt.check_induction_prompts(run_study.make_quizzes(0, key), bt.released_queries(key, 0), run_study.for_model(key))
    print(f"{key}: prompts consistent with the released ICLR prompts")

## Run the induction study

The three arms of one seed are sent together. One YAML per replicate is written to the
bucket as `induction/<model>/seed=<seed>/<arm>--<timestamp>.yaml` below the prefix, the
released-results layout; with `RESULTS_S3 = None` it is written to `results/` instead. When
the run ends, the bucket's replicates are mirrored to
`results/induction/induction/<model>/seed=<seed>/`, which is where the report reads them.

In [ ]:
harness = bt.induction_harness(list(TRIO.values()), INDUCTION_SEEDS, results_dir=RESULTS, s3_uri=RESULTS_S3)
for key, model in TRIO.items():
    print(f"== {key}")
    bt.run_induction(harness, model, REGION, REASONING[key], max_parallel=CONCURRENCY)
if RESULTS_S3:
    print(f"mirrored {bt.pull_induction(RESULTS_S3, RESULTS)} files to {RESULTS / 'induction'}")

## Run the Horn study

Each model's rung is rendered with all eight arms and its paper arms are checked against the
ICLR digests, then every (arm, seed, replicate) cell is sent and scored with the ICLR answer
extraction. Rows are appended to `results/horn/<model>.jsonl` and, when `RESULTS_S3` is set,
pushed to the bucket as `deduction/horn/<model>/rows--<stamp>.jsonl` parts (every 200 new
rows, and when the run ends or is interrupted). Before it picks the cells to run, a model's
run pulls the bucket's rows, so rows collected on any machine are not sent again. A transport
failure is written as an `exception` row and is retried on the next run.

In [ ]:
for key, model in TRIO.items():
    m = bt.horn_m(key)
    rung = bt.render_horn_rung(m, HORN_SEEDS, arms=HORN_ARMS)
    rows = bt.run_horn(model, REGION, REASONING[key], rung, HORN_SEEDS, HORN_REPS,
                       concurrency=CONCURRENCY, out=bt.horn_rows_path(key, RESULTS),
                       arms=HORN_ARMS, s3_uri=RESULTS_S3)
    verdicts = {}
    for r in rows:
        verdicts[r["verdict"]] = verdicts.get(r["verdict"], 0) + 1
    print(f"== {key} m{m}: {len(rows)} new rows {verdicts}")

## Results

When `RESULTS_S3` is set, the first cell below pulls both studies from the bucket into the
local mirror, so a fresh checkout can report from the bucket alone: it needs the config cell
and AWS credentials for the bucket, nothing from Bedrock.

Accuracies and pass rates are shown next to the published ICLR values. Only `intens` is
like-for-like with the published induction values (see the `extens` note above). For Horn,
`horn_repro.report` gives the contrasts (`lem-both`, `pad-both`, `disc-both`) for the four
paper arms only; the four review arms (`fwd`, `tjunk`, `rep`, `marked-lemma`) appear in the
per-arm pass rates and in the per-arm breakdown. The breakdowns show how many answers were
truncated or could not be graded, and how many responses came back without reasoning, which
must be zero.

In [ ]:
if RESULTS_S3:
    print(f"induction: {bt.pull_induction(RESULTS_S3, RESULTS)} files written to {RESULTS / 'induction'}")
    for k in MODELS:
        print(f"horn {k}: {bt.pull_horn_rows(k, RESULTS_S3, RESULTS)} rows appended")
else:
    print("RESULTS_S3 is None: reporting from the local results")

In [ ]:
print(induction_repro.report(RESULTS / "induction"))

In [ ]:
rows_files = [bt.horn_rows_path(k, RESULTS) for k in MODELS if bt.horn_rows_path(k, RESULTS).exists()]
print(horn_repro.report(rows_files))

In [ ]:
ind = bt.induction_reasoning_report(RESULTS)
horn = bt.horn_breakdown(rows_files)
print("induction (model, arm): counts")
for k, v in sorted(ind.items()):
    print(f"  {k}: {v}")
print("horn (model, arm): counts")
for k, v in sorted(horn.items()):
    print(f"  {k}: {v}")
unreasoned = {k: v["unreasoned"] for k, v in {**ind, **horn}.items() if v.get("unreasoned")}
assert not unreasoned, f"responses without reasoning: {unreasoned}"